# Data-derived linear-interpolation sentencing analysis

This notebook has no scikit-learn model. It learns per-drug, monotonic piecewise-linear curves from verified training cases, follows the legacy weighted-total rule for mixed drugs, and derives the full sentencing path with training-only direct factor effects.

In [1]:
from linear_interpolation_model import run_analysis

REFRESH_CACHE = False
results = run_analysis(refresh_cache=REFRESH_CACHE)
print(f"Wrote {results['output_path']}")

Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/linear_interpolation_analysis.xlsx


## Results guide

All generic parameters are fitted on the training partition only. Document-level excluded judgments may contribute only to role and circumstance fitting; they are omitted from drug curves, generic stage effects, held-out predictions, and non-role metrics. The legacy `Role of the defendant` factor is retained as factual data but has no model effect; only the dedicated sentencing-role profile can adjust the role stage. Coverage is reported alongside error: a case containing a drug type without a supported curve is deliberately excluded from interpolation error rather than receiving a zero-month prediction.

### Eligibility summary

- `scope`: what is being counted, such as judgments, trials, eligible single-drug starting-point rows, direct adjustments, or workbook exclusions.
- `partition`: `train`, `test`, or `all`.
- `count`: number of items in that scope and partition.

### Held-out metrics

- `stage`: point in the sentencing calculation being evaluated.
- `all_test_trials`: all trials assigned to the held-out partition.
- `covered_test_trials`: held-out trials with both an actual value and a supported prediction at that stage.
- `coverage_rate`: `covered_test_trials / all_test_trials`.
- `mae_months`: mean absolute prediction error, in months, among covered trials.
- `median_absolute_error_months`: median absolute prediction error, in months.
- `accuracy_eligible_test_trials`: covered trials with a positive actual sentence, which can be scored by the case-accuracy formula.
- `mean_case_accuracy`: mean of `max(1 - abs(actual - predicted) / actual, 0)`.
- `mean_case_accuracy_percent`: the same accuracy on a 0–100% scale.

### Method comparison

- `method`: the model being compared.
- `common_test_trials`: only held-out trials supported by both methods, ensuring a like-for-like comparison.
- `starting_point_mae_months` and `final_sentence_mae_months`: mean absolute error in months at those stages.

### Legacy-percentage comparison

- `method`: data-derived medians or mapped legacy percentages.
- `common_compatible_test_trials`: held-out trials where both approaches can make the same complete prediction.
- `final_sentence_mae_months` and `median_absolute_error_months`: final-sentence error in months.

### Curve support

- `drug_type`: drug being considered for a starting-point curve.
- `supported`: whether there are enough single-drug training trials to fit that curve.
- `reason`: why a curve is unavailable, when unsupported.
- `training_trials` and `training_judgments`: supporting training sample sizes.

### Factor percentage errors

- `stage` and `direction`: whether the adjustment increases or reduces the relevant sentencing stage.
- `canonical_factor`: normalized factor name used by the model.
- `model_status`: whether the factor had enough training evidence for a learned effect.
- `training_direct_adjustments`, `model_percentage`, and `model_median_months`: the training support and learned median adjustment, expressed as a percentage of its stage base and in months.
- `test_direct_adjustments` and `test_judgments`: held-out evidence used to assess the learned effect.
- `actual_median_percentage`: median observed held-out adjustment as a percentage of its stage base.
- `mae_months`, `median_absolute_error_months`, and `mean_signed_error_months`: held-out error in months; a negative signed error indicates under-prediction.
- `mae_percentage_points`, `mean_signed_error_percentage_points`, and `mean_absolute_percentage_error`: the same comparison in percentage terms.

### Unsupported test drugs

- `case_id` and `neutral_citation`: judgment identifiers.
- `trial_index`, `charge_no`, and `defendant_id`: the individual trial key.
- `starting_prediction_status`: why no supported starting-point prediction was available.
- `unsupported_drugs`: drug types without a supported curve.
- `drugs_json`: source drug quantities and extraction details for review.

### Role-base and circumstance effects

The role-base table is fitted from training data only. `component` identifies a primary role, pooled supplementary circumstance, or the pooled severe-role cross-border effect. `name` is the user-facing selection. `support_trials` is the number of direct labelled training trials. `effect_fraction` is added to the starting point (for example, `0.05` means a 5% enhancement); `Courier / Storekeeper` is always zero. `status` explains whether the effect is supported, fixed, or unavailable.

In [2]:
display(results['eligibility_summary'])
display(results['role_effect_support'])
display(results['metrics'])
display(results['legacy_percentage_comparison'])
display(results['curve_support'])
display(results['factor_percentage_errors'])
display(results['unsupported_test_drugs'].head(20))

,scope,partition,count
0,judgments,train,1617
1,judgments,test,403
2,trials,train,1989
3,trials,test,489
4,starting-point eligible single-drug training t...,train,1085
5,direct factor adjustments,train,2335
6,source-excluded trials reserved for role and c...,all,385
7,excluded role-workbook trials removed from dev...,all,141


,component,name,support_trials,effect_fraction,status
0,primary role,Courier / Storekeeper,0,0.000000,fixed zero
1,primary role,Actual trafficker,65,0.050633,supported
2,primary role,Manager / Organiser,14,0.061250,supported
3,primary role,Operator / Financial controller,1,0.077922,supported
4,supplementary circumstance,Divan keeping,6,0.079122,supported
5,supplementary circumstance,Manufacturing,5,0.091954,supported
6,severe-role cross-border circumstance,Cross-border trafficking,3,0.078125,supported


,stage,all_test_trials,covered_test_trials,coverage_rate,mae_months,median_absolute_error_months,accuracy_eligible_test_trials,mean_case_accuracy,mean_case_accuracy_percent
0,Starting point,489,421,0.860941,8.965769,3.488038,420,0.917087,91.708747
1,Sentence after role,489,437,0.893661,10.201034,3.979768,436,0.912093,91.209344
2,Notional sentence,489,462,0.944785,10.210153,3.988309,461,0.915347,91.534716
3,Non-plea mitigation reduction,489,155,0.316973,7.197893,1.907143,136,0.440690,44.069003
4,Final sentence,489,462,0.944785,8.086178,2.939120,461,0.902093,90.209335


,method,common_compatible_test_trials,final_sentence_mae_months,median_absolute_error_months
0,data-derived median effects,204,7.769424,2.834941
1,legacy-mapped hard-coded percentages,204,8.142443,3.092173


,drug_type,supported,reason,training_trials,training_judgments
0,Cannabis,True,None,40,32
1,Cathinones,False,fewer than 10 training trials,0,0
2,Cocaine,True,None,551,459
3,Ecstasy,True,None,10,9
4,Etomidate,False,fewer than 10 training trials,0,0
5,Fluorodeschloroketamine,False,fewer than 10 training trials,5,3
6,GHB/GBL,False,fewer than 10 training trials,0,0
7,Heroin,True,None,91,76
8,Ketamine,True,None,158,135
9,Methamphetamine,True,None,226,200


,stage,direction,canonical_factor,model_status,training_direct_adjustments,model_percentage,model_median_months,test_direct_adjustments,test_judgments,actual_median_percentage,mae_months,median_absolute_error_months,mean_signed_error_months,mae_percentage_points,mean_signed_error_percentage_points,mean_absolute_percentage_error
0,aggravation,increase,CSD supervision,unsupported: zero contribution,0,0.000000,0.000000,1,1,4.761905,12.000000,1.200000e+01,-12.000000,4.761905,-4.761905,100.000000
1,aggravation,increase,Cross-border trafficking,supported,104,6.079084,12.000000,21,21,7.228916,5.149641,3.604952e+00,-1.139980,2.609916,-1.034236,31.395777
2,aggravation,increase,Illegal immigrant,supported,4,8.029653,8.500000,2,2,9.642857,2.222669,2.222669e+00,-0.404877,4.642857,-1.613204,52.192743
3,aggravation,increase,Multiple drugs,supported,301,3.846154,3.000000,79,77,4.347826,2.921558,9.615385e-01,-1.244051,4.836447,-3.253478,64.084604
4,aggravation,increase,On bail,supported,40,4.412719,3.000000,13,13,3.333333,1.495682,9.714471e-01,0.808316,2.381373,0.222572,42.052583
5,aggravation,increase,Other,supported,38,5.358852,3.000000,16,8,4.905437,4.620215,1.358373e+00,0.007536,5.341368,-3.027470,60.330144
6,aggravation,increase,Persistent offender,supported,205,4.000000,3.000000,45,45,4.545455,2.321102,1.400000e+00,0.260942,2.541577,-0.360144,60.257987
7,aggravation,increase,Refugee claimant,supported,42,6.596342,6.000000,9,8,8.333333,2.433569,2.416878e+00,-1.866574,4.146970,-3.325238,44.164937
8,aggravation,increase,Suspended sentence,supported,4,5.146628,2.000000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,aggravation,increase,Use of minors,supported,13,4.615385,5.000000,1,1,7.407407,4.523077,4.523077e+00,-4.523077,2.792023,-2.792023,37.692308


,case_id,neutral_citation,trial_index,charge_no,defendant_id,starting_prediction_status,unsupported_drugs,drugs_json
65,[2021] HKDC 1464,[2021] HKDC 1464,0,1,1,unsupported drug curve,Fluorodeschloroketamine | Other,"[{""drug_type"": ""Ketamine"", ""other_drug_type"": ..."
144,[2021] HKDC 734,[2021] HKDC 734,0,1,1,unsupported drug curve,Other,"[{""drug_type"": ""Heroin"", ""other_drug_type"": nu..."
194,[2021] HKDC 970,[2021] HKDC 970,0,1,1,unsupported drug curve,Other,"[{""drug_type"": ""Cannabis"", ""other_drug_type"": ..."
522,[2021] HKCFI 1919,[2021] HKCFI 1919,2,3,1,no positive drug quantity,,"[{""drug_type"": ""Other"", ""other_drug_type"": ""Un..."
631,[2025] HKDC 880,[2025] HKDC 880,0,2,1,unsupported drug curve,Morphine | Other,"[{""drug_type"": ""Heroin"", ""other_drug_type"": nu..."
662,[2024] HKDC 961,[2024] HKDC 961,1,2,1,unsupported drug curve,Other | THC/CBD,"[{""drug_type"": ""Cannabis"", ""other_drug_type"": ..."
671,[2024] HKDC 1963,[2024] HKDC 1963,0,1,1,unsupported drug curve,THC/CBD,"[{""drug_type"": ""Cannabis"", ""other_drug_type"": ..."
710,[2025] HKCFI 1365,[2025] HKCFI 1365,0,1,1,unsupported drug curve,Other,"[{""drug_type"": ""Heroin"", ""other_drug_type"": nu..."
862,[2025] HKDC 1407,[2025] HKDC 1407,0,1,1,unsupported drug curve,Other,"[{""drug_type"": ""Heroin"", ""other_drug_type"": nu..."
939,[2022] HKDC 1239,[2022] HKDC 1239,0,1,1,unsupported drug curve,Other,"[{""drug_type"": ""Other"", ""other_drug_type"": ""Mi..."
